<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **n8n: workflows, triggers, nodos y primeros flujos low-code**
## *De scripts aislados a workflows orquestados*

En este laboratorio empezamos el bloque de automatización low-code.

La idea central será:

> **un workflow conecta eventos, datos, decisiones y acciones**

Trabajaremos con el modelo mental de n8n:

```text
Trigger
   ↓
Datos
   ↓
Transformación
   ↓
Decisión
   ↓
Acción
   ↓
Resultado
```

Aunque n8n se ejecuta fuera del notebook, utilizaremos Python para:

- generar datos de prueba;
- entender el formato de los items;
- simular transformaciones;
- validar payloads;
- construir ejemplos de expresiones;
- preparar archivos JSON;
- crear payloads para webhooks;
- diseñar workflows reproducibles.

## Objetivos

Al finalizar deberías poder:

1. Explicar qué es un workflow.
2. Diferenciar trigger y action.
3. Entender qué es un nodo.
4. Comprender cómo viajan los items entre nodos.
5. Utilizar Manual Trigger, Schedule Trigger y Webhook conceptualmente.
6. Entender el nodo HTTP Request.
7. Entender Edit Fields (Set).
8. Utilizar expresiones como `{{$json.campo}}`.
9. Diseñar condiciones con IF.
10. Diseñar múltiples rutas con Switch.
11. Entender Merge.
12. Utilizar Code para lógica puntual.
13. Diseñar un flujo con control de errores.
14. Preparar datos para un webhook.
15. Construir un primer workflow de automatización real.

## 1. Preparación

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path(
    "demo_n8n"
)

if RAIZ.exists():
    shutil.rmtree(
        RAIZ
    )

for carpeta in [
    RAIZ / "data",
    RAIZ / "payloads",
    RAIZ / "workflows",
    RAIZ / "outputs"
]:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Entorno creado:",
    RAIZ.resolve()
)

## 2. ¿Qué es n8n?

n8n es una plataforma de automatización basada en **workflows de nodos conectados**.

Cada nodo realiza una responsabilidad.

Ejemplo:

```text
Webhook
   ↓
Validar datos
   ↓
IF
 ┌─┴────────────┐
 ↓              ↓
válido         inválido
 ↓              ↓
guardar        registrar error
```

El valor principal no es “evitar Python”, sino **orquestar componentes**.

## 3. Script tradicional vs workflow

In [ ]:
comparativa = pd.DataFrame({
    "criterio": [
        "Representación",
        "Trigger",
        "Conectores",
        "Control visual",
        "Lógica compleja",
        "Reutilización"
    ],
    "Python": [
        "Código",
        "Cron / servicio / evento",
        "Librerías / APIs",
        "Indirecto",
        "Muy flexible",
        "Funciones / paquetes"
    ],
    "n8n": [
        "Grafo de nodos",
        "Nodos trigger",
        "Integraciones",
        "Visual",
        "Nodos + Code",
        "Sub-workflows"
    ]
})

comparativa

### Interpretación

Python y n8n son complementarios.

Una arquitectura frecuente será:

```text
n8n
 ↓
orquestación
 ↓
Python
 ↓
lógica especializada
```

## 4. Anatomía de un workflow

In [ ]:
componentes = pd.DataFrame({
    "elemento": [
        "Trigger",
        "Node",
        "Connection",
        "Item",
        "Expression",
        "Execution"
    ],
    "funcion": [
        "Inicia el flujo",
        "Realiza una operación",
        "Une nodos",
        "Unidad de datos",
        "Referencia valores dinámicos",
        "Una ejecución concreta"
    ]
})

componentes

## 5. Triggers

In [ ]:
triggers = pd.DataFrame({
    "trigger": [
        "Manual Trigger",
        "Schedule Trigger",
        "Webhook",
        "App/Event Trigger"
    ],
    "uso": [
        "Pruebas manuales",
        "Ejecución periódica",
        "Evento HTTP",
        "Evento de una integración"
    ],
    "ejemplo": [
        "Probar flujo",
        "Cada mañana",
        "Formulario recibido",
        "Nuevo registro"
    ]
})

triggers

## 6. Datos: los items

n8n procesa datos como items.

Conceptualmente podemos imaginar:

```json
[
  {
    "json": {
      "cliente": "Empresa A",
      "importe": 1200
    }
  },
  {
    "json": {
      "cliente": "Empresa B",
      "importe": 800
    }
  }
]
```

Cada nodo recibe items y genera items de salida.

In [ ]:
items_n8n = [
    {
        "json": {
            "cliente":
                "Empresa A",
            "importe":
                1200,
            "email":
                "a@example.com"
        }
    },
    {
        "json": {
            "cliente":
                "Empresa B",
            "importe":
                800,
            "email":
                "b@example.com"
        }
    }
]

items_n8n

## 6.1 Convertir items a DataFrame

In [ ]:
df_items = pd.DataFrame(
    [
        item["json"]
        for item
        in items_n8n
    ]
)

df_items

### Interpretación

El objeto `json` contiene normalmente los datos que utilizaremos mediante expresiones.

## 7. Expresiones

In [ ]:
expresiones = pd.DataFrame({
    "expresion": [
        "{{$json.cliente}}",
        "{{$json.importe}}",
        "{{$json.email}}",
        "{{$now}}"
    ],
    "significado": [
        "Campo cliente del item actual",
        "Campo importe",
        "Campo email",
        "Fecha/hora actual"
    ]
})

expresiones

## 7.1 Simular resolución de expresiones

In [ ]:
item_actual = items_n8n[0][
    "json"
]

print(
    "{{$json.cliente}}",
    "→",
    item_actual[
        "cliente"
    ]
)

print(
    "{{$json.importe}}",
    "→",
    item_actual[
        "importe"
    ]
)

## 8. Edit Fields (Set)

El nodo **Edit Fields (Set)** permite:

- crear campos;
- renombrar;
- seleccionar qué campos continúan;
- construir estructuras.

Ejemplo conceptual:

```text
entrada:
cliente = Empresa A
importe = 1200

salida:
cliente = Empresa A
importe = 1200
prioridad = alta
```

## 8.1 Simular Edit Fields

In [ ]:
def edit_fields_simulado(
    item
):
    nuevo = item.copy()

    nuevo[
        "prioridad"
    ] = (
        "alta"
        if nuevo[
            "importe"
        ] >= 1000
        else "normal"
    )

    return nuevo


df_editado = pd.DataFrame(
    [
        edit_fields_simulado(
            item["json"]
        )
        for item
        in items_n8n
    ]
)

df_editado

## 9. IF: bifurcación

In [ ]:
def nodo_if(
    item
):
    return (
        item[
            "importe"
        ] >= 1000
    )


rama_true = []
rama_false = []

for item in df_editado.to_dict(
    orient="records"
):
    if nodo_if(
        item
    ):
        rama_true.append(
            item
        )
    else:
        rama_false.append(
            item
        )

print(
    "TRUE:",
    rama_true
)

print(
    "FALSE:",
    rama_false
)

### Interpretación

El nodo IF no transforma necesariamente el dato: decide por qué salida continúa.

## 10. Switch: múltiples ramas

In [ ]:
casos = pd.DataFrame({
    "cliente": [
        "A",
        "B",
        "C",
        "D"
    ],
    "importe": [
        300,
        900,
        1500,
        5000
    ]
})

def nivel_importe(
    importe
):
    if importe < 500:
        return "bajo"

    if importe < 1500:
        return "medio"

    if importe < 3000:
        return "alto"

    return "critico"


casos[
    "ruta"
] = casos[
    "importe"
].apply(
    nivel_importe
)

casos

## 10.1 Visualizar rutas

In [ ]:
conteo = (
    casos[
        "ruta"
    ]
    .value_counts()
)

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    conteo.index,
    conteo.values
)

plt.ylabel(
    "Items"
)

plt.title(
    "Distribución de ramas Switch"
)

plt.tight_layout()
plt.show()

## 11. Merge

Merge permite volver a combinar ramas.

Ejemplo:

```text
           ┌→ rama A ─┐
entrada ───┤          ├→ Merge
           └→ rama B ─┘
```

El comportamiento exacto depende del modo de combinación seleccionado.

## 11.1 Simular unión de ramas

In [ ]:
rama_a = pd.DataFrame({
    "id": [
        1,
        2
    ],
    "estado": [
        "procesado",
        "procesado"
    ]
})

rama_b = pd.DataFrame({
    "id": [
        3,
        4
    ],
    "estado": [
        "revision",
        "revision"
    ]
})

pd.concat(
    [
        rama_a,
        rama_b
    ],
    ignore_index=True
)

## 12. HTTP Request

El nodo HTTP Request permite consumir APIs sin escribir directamente `requests`.

Conceptualmente configura:

```text
Method: GET
URL: https://api.ejemplo.com/items
Headers: ...
Query parameters: ...
Authentication: ...
```

Su salida puede alimentar nodos posteriores.

## 12.1 Preparar payload para una API

In [ ]:
payload_api = {
    "cliente":
        "Empresa A",
    "importe":
        1200,
    "prioridad":
        "alta"
}

ruta_payload = (
    RAIZ
    / "payloads"
    / "payload_api.json"
)

ruta_payload.write_text(
    json.dumps(
        payload_api,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_payload.read_text(
        encoding="utf-8"
    )
)

## 13. Webhook

Un Webhook Trigger convierte una petición HTTP en un evento que inicia el workflow.

Ejemplo:

```text
Formulario
   ↓ POST
Webhook n8n
   ↓
workflow
```

El payload podría ser:

```json
{
  "cliente": "Empresa A",
  "importe": 1200
}
```

## 13.1 Generar payload webhook

In [ ]:
payload_webhook = {
    "cliente_id":
        "C001",
    "cliente":
        "Empresa Norte",
    "email":
        "contacto@example.com",
    "importe":
        2500,
    "tipo":
        "pedido"
}

ruta_webhook = (
    RAIZ
    / "payloads"
    / "webhook_demo.json"
)

ruta_webhook.write_text(
    json.dumps(
        payload_webhook,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_webhook.read_text(
        encoding="utf-8"
    )
)

## 14. Validar payload antes de procesarlo

In [ ]:
CAMPOS_WEBHOOK = {
    "cliente_id",
    "cliente",
    "email",
    "importe",
    "tipo"
}


def validar_payload(
    payload
):
    errores = []

    faltantes = (
        CAMPOS_WEBHOOK
        - set(
            payload.keys()
        )
    )

    if faltantes:
        errores.append(
            f"Faltan campos: {sorted(faltantes)}"
        )

    if (
        "importe"
        in payload
        and payload[
            "importe"
        ] < 0
    ):
        errores.append(
            "importe no puede ser negativo"
        )

    return errores


validar_payload(
    payload_webhook
)

## 14.1 Provocar payload inválido

In [ ]:
payload_invalido = {
    "cliente_id":
        "C002",
    "cliente":
        "Empresa Sur",
    "importe":
        -100
}

validar_payload(
    payload_invalido
)

### Interpretación

Un webhook no debe asumir que todos los datos recibidos son correctos.

El workflow debe validar antes de ejecutar acciones.

## 15. Code node

El nodo **Code** permite introducir JavaScript cuando la lógica resulta incómoda con nodos visuales.

Ejemplo conceptual:

```javascript
return items.map(item => {
  item.json.total = item.json.unidades * item.json.precio;
  return item;
});
```

Debe utilizarse para lógica puntual, evitando convertir todo el workflow en un único bloque de código.

## 15.1 Simular Code node en Python

In [ ]:
ventas = pd.DataFrame({
    "producto": [
        "A",
        "B",
        "C"
    ],
    "unidades": [
        2,
        5,
        3
    ],
    "precio": [
        100,
        20,
        50
    ]
})

ventas[
    "total"
] = (
    ventas[
        "unidades"
    ]
    * ventas[
        "precio"
    ]
)

ventas

## 16. Schedule Trigger

Un Schedule Trigger permite iniciar workflows periódicamente.

Ejemplos:

```text
Cada día a las 08:00
Cada lunes
Cada hora
Cada mes
```

Un caso típico de este curso:

```text
Schedule Trigger
       ↓
API
       ↓
procesar
       ↓
generar informe
       ↓
enviar / guardar
```

## 17. Workflow manual de ejemplo

Vamos a diseñar:

```text
Manual Trigger
      ↓
Edit Fields
      ↓
IF importe >= 1000
   ┌──┴───┐
 true    false
  ↓        ↓
Alta     Normal
  └──┬─────┘
    Merge
      ↓
Salida
```

## 17.1 Representar workflow como tabla

In [ ]:
workflow_conceptual = pd.DataFrame({
    "orden": [
        1,
        2,
        3,
        4,
        5,
        6
    ],
    "nodo": [
        "Manual Trigger",
        "Edit Fields",
        "IF",
        "Alta prioridad",
        "Prioridad normal",
        "Merge"
    ],
    "responsabilidad": [
        "Iniciar",
        "Preparar datos",
        "Decidir",
        "Ruta true",
        "Ruta false",
        "Reunir"
    ]
})

workflow_conceptual

## 18. Diseño de workflow con Webhook

In [ ]:
flujo_webhook = pd.DataFrame({
    "paso": [
        1,
        2,
        3,
        4,
        5,
        6
    ],
    "nodo": [
        "Webhook",
        "Edit Fields",
        "IF",
        "HTTP Request",
        "Edit Fields",
        "Respond to Webhook"
    ],
    "descripcion": [
        "Recibir payload",
        "Normalizar",
        "Validar condición",
        "Llamar servicio",
        "Preparar respuesta",
        "Responder"
    ]
})

flujo_webhook

## 19. Estado y trazabilidad

In [ ]:
ejecuciones = pd.DataFrame({
    "execution_id": [
        "E001",
        "E002",
        "E003",
        "E004"
    ],
    "estado": [
        "success",
        "success",
        "error",
        "success"
    ],
    "segundos": [
        0.8,
        1.2,
        3.5,
        0.9
    ]
})

ejecuciones

## 19.1 KPI de ejecuciones

In [ ]:
kpis = {
    "ejecuciones":
        len(
            ejecuciones
        ),

    "success":
        int(
            (
                ejecuciones[
                    "estado"
                ]
                == "success"
            ).sum()
        ),

    "error":
        int(
            (
                ejecuciones[
                    "estado"
                ]
                == "error"
            ).sum()
        ),

    "tiempo_medio":
        float(
            ejecuciones[
                "segundos"
            ].mean()
        )
}

kpis[
    "tasa_exito"
] = (
    kpis[
        "success"
    ]
    / kpis[
        "ejecuciones"
    ]
)

kpis

## 19.2 Visualizar

In [ ]:
estado = (
    ejecuciones[
        "estado"
    ]
    .value_counts()
)

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    estado.index,
    estado.values
)

plt.ylabel(
    "Ejecuciones"
)

plt.title(
    "Estado de workflows"
)

plt.tight_layout()
plt.show()

## 20. Manejo de errores

En un workflow debemos decidir:

```text
error
 ↓
¿reintentable?
 ├─ sí → retry
 └─ no → registrar / alertar / ruta de error
```

Errores frecuentes:

- API caída;
- timeout;
- credenciales;
- JSON inesperado;
- dato obligatorio ausente;
- rate limit;
- error de negocio.

## 20.1 Clasificar errores

In [ ]:
errores = pd.DataFrame({
    "error": [
        "Timeout API",
        "401 Unauthorized",
        "Campo email ausente",
        "429 Too Many Requests",
        "Importe negativo"
    ],
    "tipo": [
        "tecnico",
        "tecnico",
        "datos",
        "tecnico",
        "negocio"
    ],
    "reintentar": [
        True,
        False,
        False,
        True,
        False
    ]
})

errores

## 21. Diseñar un flujo de error

In [ ]:
flujo_error = pd.DataFrame({
    "paso": [
        1,
        2,
        3,
        4
    ],
    "accion": [
        "Detectar fallo",
        "Clasificar",
        "Registrar",
        "Reintentar o escalar"
    ]
})

flujo_error

## 22. Variables y credenciales

En n8n debemos evitar introducir secretos directamente en nodos o expresiones cuando exista un mecanismo de credenciales.

Ejemplos de secretos:

- API keys;
- tokens;
- passwords;
- OAuth credentials.

Conceptualmente:

```text
workflow
   ↓
credential reference
   ↓
secret
```

No:

```text
workflow JSON
   ↓
token escrito manualmente
```

## 23. Ejemplo de datos entre nodos

In [ ]:
datos_nodos = pd.DataFrame({
    "nodo": [
        "Webhook",
        "Edit Fields",
        "IF",
        "HTTP Request",
        "Respond"
    ],
    "entrada": [
        "HTTP request",
        "payload",
        "payload normalizado",
        "item válido",
        "respuesta servicio"
    ],
    "salida": [
        "item",
        "item normalizado",
        "true/false",
        "JSON externo",
        "HTTP response"
    ]
})

datos_nodos

## 24. Detectar una mala arquitectura

In [ ]:
workflow_malo = pd.DataFrame({
    "nodo": [
        "Webhook",
        "Code gigante",
        "Respond"
    ],
    "lineas_logica": [
        0,
        250,
        0
    ]
})

workflow_malo

### Problema

Si toda la lógica se concentra en un único Code node:

- perdemos legibilidad visual;
- dificultamos mantenimiento;
- dificultamos debugging;
- desaprovechamos nodos especializados.

Conviene utilizar Code para lógica que realmente lo necesite.

## 25. Refactor conceptual

In [ ]:
workflow_refactor = pd.DataFrame({
    "nodo": [
        "Webhook",
        "Edit Fields",
        "IF",
        "HTTP Request",
        "Code pequeño",
        "Respond"
    ],
    "responsabilidad": [
        "Entrada",
        "Normalización",
        "Decisión",
        "Servicio",
        "Transformación especial",
        "Respuesta"
    ]
})

workflow_refactor

## 26. Primer mini-proyecto: clasificación de solicitudes

Construiremos conceptualmente:

```text
Webhook
   ↓
solicitud
   ↓
validar campos
   ↓
Switch tipo
 ┌────┼────┐
 ↓    ↓    ↓
venta soporte otro
 ↓    ↓    ↓
acciones específicas
 └────┼────┘
      ↓
responder
```

## 26.1 Datos de solicitudes

In [ ]:
solicitudes = pd.DataFrame({
    "id": [
        1,
        2,
        3,
        4,
        5
    ],
    "tipo": [
        "venta",
        "soporte",
        "venta",
        "otro",
        "soporte"
    ],
    "prioridad": [
        2,
        5,
        4,
        1,
        3
    ],
    "email": [
        "a@example.com",
        "b@example.com",
        "c@example.com",
        "d@example.com",
        "e@example.com"
    ]
})

solicitudes

## 26.2 Simular Switch

In [ ]:
rutas = {}

for tipo, grupo in solicitudes.groupby(
    "tipo"
):
    rutas[
        tipo
    ] = grupo.to_dict(
        orient="records"
    )

rutas

## 27. Preparar payloads para probar Webhook

In [ ]:
for _, fila in solicitudes.iterrows():

    payload = fila.to_dict()

    ruta = (
        RAIZ
        / "payloads"
        / f'solicitud_{fila["id"]}.json'
    )

    ruta.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

print(
    "Payloads:",
    len(
        list(
            (
                RAIZ
                / "payloads"
            ).glob(
                "solicitud_*.json"
            )
        )
    )
)

## 28. Probar Webhook desde terminal

Cuando tengas un Webhook de prueba activo en n8n, puedes enviar un payload con `curl`.

Ejemplo:

```bash
curl -X POST "URL_WEBHOOK_DE_PRUEBA"   -H "Content-Type: application/json"   -d '{"id":1,"tipo":"venta","prioridad":2,"email":"a@example.com"}'
```

También podrás hacerlo desde Python en el siguiente laboratorio.

## 29. Comprobar herramientas locales

In [ ]:
herramientas = {
    "docker":
        shutil.which(
            "docker"
        ),

    "n8n":
        shutil.which(
            "n8n"
        ),

    "node":
        shutil.which(
            "node"
        ),

    "npm":
        shutil.which(
            "npm"
        )
}

herramientas

## 30. Opciones de ejecución de n8n

Dependiendo del entorno del curso, n8n puede ejecutarse:

### n8n Cloud

Sin instalación local.

### Instalación con npm

```bash
npx n8n
```

### Docker

Ejemplo conceptual:

```bash
docker run --rm -it   -p 5678:5678   -v n8n_data:/home/node/.n8n   docker.n8n.io/n8nio/n8n
```

Después se accede normalmente desde el navegador al servicio local configurado.

## 31. Crear guía de workflow reproducible

In [ ]:
guia_workflow = {
    "nombre":
        "Clasificador de solicitudes",

    "trigger":
        "Webhook",

    "nodos": [
        "Webhook",
        "Edit Fields",
        "Switch",
        "Edit Fields Venta",
        "Edit Fields Soporte",
        "Edit Fields Otro",
        "Respond to Webhook"
    ],

    "campos_entrada": [
        "id",
        "tipo",
        "prioridad",
        "email"
    ],

    "rutas_switch": [
        "venta",
        "soporte",
        "otro"
    ]
}

ruta_guia = (
    RAIZ
    / "workflows"
    / "clasificador_solicitudes_spec.json"
)

ruta_guia.write_text(
    json.dumps(
        guia_workflow,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    ruta_guia.read_text(
        encoding="utf-8"
    )
)

### Nota

Este JSON es una **especificación pedagógica del flujo**, no pretende sustituir el JSON exportado por n8n.

Una vez creado el workflow en la interfaz, conviene exportarlo desde n8n y guardarlo junto al proyecto.

## 32. Checklist de workflow

- [ ] ¿El trigger es el adecuado?
- [ ] ¿La entrada está documentada?
- [ ] ¿Conozco el formato de los items?
- [ ] ¿Las expresiones usan los campos correctos?
- [ ] ¿Se validan campos obligatorios?
- [ ] ¿IF/Switch representan decisiones claras?
- [ ] ¿HTTP Request tiene timeout/configuración razonable?
- [ ] ¿Las credenciales están gestionadas correctamente?
- [ ] ¿Los errores están contemplados?
- [ ] ¿El Code node es pequeño?
- [ ] ¿El flujo devuelve una respuesta clara?
- [ ] ¿Se puede probar con datos reproducibles?
- [ ] ¿El workflow puede exportarse?

## 33. Mini-práctica integrada

Construye en n8n un workflow:

```text
Manual Trigger
      ↓
Edit Fields
      ↓
3 registros de ejemplo
      ↓
IF importe >= 1000
  ┌─────────────┐
  ↓             ↓
prioridad=alta  prioridad=normal
  └──────┬──────┘
         ↓
       Merge
         ↓
      salida
```

Después:

1. ejecútalo;
2. inspecciona cada nodo;
3. modifica un importe;
4. observa cómo cambia la rama;
5. provoca un campo ausente;
6. corrígelo;
7. exporta el workflow.

## 34. Práctica Webhook

Crea otro workflow:

```text
Webhook POST
    ↓
Edit Fields
    ↓
Switch tipo
 ┌────┼────┐
 ↓    ↓    ↓
venta soporte otro
 └────┼────┘
      ↓
Respond to Webhook
```

Prueba los payloads generados por este notebook.

## 35. Ejercicios finales

### Ejercicio 1
Crea un Manual Trigger.

### Ejercicio 2
Añade Edit Fields.

### Ejercicio 3
Crea tres items.

### Ejercicio 4
Usa `{{$json.campo}}`.

### Ejercicio 5
Añade IF.

### Ejercicio 6
Prueba TRUE y FALSE.

### Ejercicio 7
Añade Switch.

### Ejercicio 8
Crea tres ramas.

### Ejercicio 9
Reúne resultados.

### Ejercicio 10
Añade HTTP Request.

### Ejercicio 11
Inspecciona JSON de respuesta.

### Ejercicio 12
Crea Webhook.

### Ejercicio 13
Envía payload por curl.

### Ejercicio 14
Provoca campo ausente.

### Ejercicio 15
Diseña ruta de error.

### Ejercicio 16
Añade un Code node pequeño.

### Ejercicio 17
Exporta workflow.

### Ejercicio 18
Reto: construye un clasificador de solicitudes completo con Webhook, Switch y respuesta.

## 36. Preguntas de reflexión

1. ¿Qué es un workflow?
2. ¿Qué es un trigger?
3. ¿Qué es un item?
4. ¿Cómo accedemos a un campo?
5. ¿Qué diferencia hay entre IF y Switch?
6. ¿Qué aporta Merge?
7. ¿Cuándo usar HTTP Request?
8. ¿Qué aporta Webhook?
9. ¿Cuándo usar Code?
10. ¿Por qué evitar un Code node gigante?
11. ¿Qué debe ocurrir con datos inválidos?
12. ¿Cómo proteger credenciales?
13. ¿Qué registrarías de una ejecución?
14. ¿Qué ventaja tiene n8n frente a scripts aislados?
15. ¿Qué ventaja mantiene Python?

## 37. Cierre

Ya tenemos el modelo mental de n8n:

```text
Trigger
  ↓
Items
  ↓
Transformación
  ↓
Decisión
  ↓
Acción
  ↓
Respuesta
```

El siguiente laboratorio conectará realmente ambos mundos:

```text
Python
   ↕
HTTP
   ↕
n8n
   ↕
Webhook / API
```

y construiremos integraciones reproducibles entre scripts Python y workflows n8n.